### MCP(model context Protocol)

- 남이 만들어둔 서버에 붙어서 도구를 받아로 수 있다.
- 공통 규격을 말한다
- 받아오는 쪽
- 비동기 : 요청을 보내고 (직원이 커피주문 받고) 다른일을 할 수 있는 
  (현재 주문 작업외에 다른 작업 진행 가능) 방식 -> async

In [37]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_postgres import PGVector, PGEngine, PGVectorStore, Column
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent
import os 

load_dotenv()   # b``.env 파일에 저장되어있는 api 키 가져오기

True

In [38]:
import asyncio
import nest_asyncio

In [39]:
# 1. model 설정
model = init_chat_model("openai:gpt-5.6-luna", reasoning_effort="none")

1. 서버 - 내컴퓨터에서 올리기
- 컴퓨터에서 별도로 올리고 외부에서 도구를 가져옴

In [40]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [41]:
local_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "/home/lee/.local/bin/uvx",
            "args": ["mcp-server-time"],
        },
        "fetch" : {

            "transport" : "stdio",
            "command" : "/home/lee/.local/bin/uvx",
            "args" : ['mcp-server-fetch']
        }
    }
)


async def get_local_tools():
    return await local_client.get_tools()


local_tools = await get_local_tools()
[tool.name for tool in local_tools]

['get_current_time', 'convert_time', 'fetch']

In [42]:
[tool for tool in local_tools]

[StructuredTool(name='get_current_time', description='Get current time in a specific timezone', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, metadata={'title': None, 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_tool_error=<function _handle_mcp_tool_error at 0x7da0959e7060>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7da0957f2de0>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no source timezone provi

In [43]:
# 1. model 설정
model = init_chat_model("openai:gpt-5.6-luna", reasoning_effort="none")

tools = local_tools

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="너는 비서다. 도구로 확인한 내용을 답하라.",
)

In [44]:
# agent 호출용 함수 - 비동기 함수
async def ask(question):
    response = await agent.ainvoke(
        {"messages": [HumanMessage(content=question)]}
    )
    return response

In [45]:
result = await ask("안녕하세요")
result["messages"]

[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='a1b64686-9d3b-4d4e-8a87-ba12b277b182'),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 497, 'total_tokens': 511, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ERA3kRj2Y5eSlOSPLDguvYtkXT8Va', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ccca-b313-7da0-900b-047e627497e8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 497, 'output_tokens': 14, 'total_tokens': 511, 'input_token_details': {'audio

In [46]:
result = await ask("현재 시간은?")
result['messages']

[HumanMessage(content='현재 시간은?', additional_kwargs={}, response_metadata={}, id='c28e8e5d-f72f-4694-a64a-580299d7cab4'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 499, 'total_tokens': 520, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ERA3lFLDyoAvVEdHyrGWnljo1K4zp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ccca-b6a9-7772-bf89-cc0c6576843f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul'}, 'id': 'call_HuMpqMmJtQhjDFL4taQV2J1b', 'type': 'tool_call'}], invalid_tool_calls

In [47]:
# 도구 추가하기 - 웹서치 - 페이지 내용 가져오기 - fetch
local_client = MultiServerMCPClient({
    "time" : {                  # 현재 시간반환하는 도구
        "transport": "stdio",
       "command": "/home/lee/.local/bin/uvx",
        "args" : ['mcp-server-time']
    },
    "fetch" : {
        "transport": "stdio", 
        "command": "/home/lee/.local/bin/uvx",
        "args" : ['mcp-server-fetch']
    }
})

async def get_local_tools():
    return await local_client.get_tools()

local_tools = await get_local_tools()
local_tools

[StructuredTool(name='get_current_time', description='Get current time in a specific timezone', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, metadata={'title': None, 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_tool_error=<function _handle_mcp_tool_error at 0x7da0959e7060>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7da0cc3763e0>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no source timezone provi

In [48]:
# 도구 추가 - 웹서치 - 페이지 네용 가져오기 - fetch

In [49]:
# 도구 추가로 추가한 후 agent 설정
agent = create_agent(
    model=model, 
    tools=local_tools,
    system_prompt="너는 비서다. 도구로 확인한 내용을 답해라. "
)

In [50]:
result = await ask("안녕하세요")
result['messages']

[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='041f2d01-1440-453e-8641-dde0d98f5c66'),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 498, 'total_tokens': 512, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ERA3puvHjucd7637sjkkBZG312231', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ccca-c5c4-76b0-9841-2515cf6fdfc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 498, 'output_tokens': 14, 'total_tokens': 512, 'input_token_details': {'audio

In [51]:
result = await ask(r"https://namu.wiki/w/Model%20Context%20Protocol?from=MCP 이 페이지 읽고 MCP 에 대해 내용 정리해줘. 웹페이지 안읽어지면 안읽어진다고 얘기해")
result['messages']

[HumanMessage(content='https://namu.wiki/w/Model%20Context%20Protocol?from=MCP 이 페이지 읽고 MCP 에 대해 내용 정리해줘. 웹페이지 안읽어지면 안읽어진다고 얘기해', additional_kwargs={}, response_metadata={}, id='53ec4809-ed57-4d3f-9a16-2f190fae3ca3'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 543, 'total_tokens': 583, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ERA3qc88z4DRz1sHLbZLoWSU30RQ5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ccca-c9b7-71f1-9a8d-e773a9429f13-0', tool_calls=[{'name': 'fetch', 'args': {'url': 'https://namu.

In [52]:
print(result['messages'][-1].content)

페이지를 읽어 내용을 확인했습니다. 다만 해당 문서는 나무위키 문서이므로, 문서 자체의 주의사항처럼 내용에 부정확하거나 최신 정보와 다른 부분이 있을 수 있습니다.

## MCP란?

**MCP(Model Context Protocol)**는 생성형 AI와 외부 데이터·도구를 연결하기 위한 **개방형 표준 프로토콜**입니다. Anthropic이 2024년 11월 25일 발표·제안했습니다.

쉽게 말하면 **“AI 시대의 API”**와 비슷한 개념입니다.

기존에는 AI 애플리케이션마다 데이터베이스, 파일, 외부 API, 자동화 도구 등에 연결하기 위한 별도의 연동 코드를 만들어야 했습니다. MCP는 이 연결 방식을 표준화해 다음과 같은 문제를 줄이는 것을 목표로 합니다.

- AI 애플리케이션과 외부 데이터 소스의 연결 방식 통일
- 서비스별·AI 모델별 맞춤형 연동 코드 감소
- 파일, 데이터베이스, 지식 베이스, API, 자동화 도구 등의 재사용성 향상
- AI 에이전트가 외부 도구를 일관된 방식으로 사용

## MCP가 필요한 이유

LLM은 학습 데이터만으로 모든 최신 정보나 사용자 전용 정보에 접근할 수 없습니다. 따라서 다음과 같은 외부 정보가 필요할 수 있습니다.

- 회사 내부 문서
- 데이터베이스
- 로컬 파일
- 최신 웹·업무 데이터
- 외부 API
- 컴퓨터 조작 및 자동화 도구

MCP를 사용하면 AI 애플리케이션이 이런 데이터와 도구에 접근하는 방식을 일정한 규격으로 통일할 수 있습니다.

예를 들어 기업이 내부 문서 검색 기능을 MCP 서버로 제공하면, 여러 AI 애플리케이션이 각자 별도의 방식으로 연동하지 않고 MCP 규격을 통해 해당 기능을 사용할 수 있습니다.

## 주요 구성 요소

### 1. MCP 호스트

MCP를 사용하는 AI 애플리케이션입니다.

예시:

- LLM 기반 챗봇
- AI 코딩 도구
- AI 에이전트
- 데스크톱 AI 애플리케이션

호스트는 사용자의 요청을 처리하고 필요한 경우 MCP 서버의 기능을 이용합니

In [ ]:
def make_agent(model, tools, system_prompt):
    agent = create_agent(
        model=model, 
        tools=tools,
        system_prompt=system_prompt
    )
    return agent

### mcp 서버가 내컴퓨터가 아닌 원격에 있는 경우
- 인터넷에  떠있는 서버에 주소만 알면 가져오서 에이전트로 구성가능
- streamble_http 방식
- deepwiki : 깃허브 저장소 문서를 읽고 질문에 대답해주는 공개 서버
- api 키가 있는 경우가 많다

In [55]:
remote_client = MultiServerMCPClient(
    {
        "deepwiki": {
            "transport": "http",
            "url": "https://mcp.deepwiki.com/mcp",
        }
    }
)


async def get_remote_tools():
    return await remote_client.get_tools()


remote_tools = await get_remote_tools()
remote_tools

[StructuredTool(name='ask_wiki_question', description="Ask any question about a GitHub repository's codebase and get an AI-powered answer\ngrounded in its DeepWiki.", args_schema={'properties': {'repoName': {'anyOf': [{'type': 'string'}, {'items': {'type': 'string'}, 'type': 'array'}], 'description': 'GitHub repository or list of repositories (max 10) in owner/repo format.'}, 'question': {'description': 'The question to ask about the repository.', 'type': 'string'}}, 'required': ['repoName', 'question'], 'type': 'object'}, metadata={'_meta': {'_fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x7da0959e7060>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7da08ee687c0>),
 StructuredTool(name='read_wiki_contents', description='View documentation about a GitHub repository.', args_schema={'properties': {'repoName': {'description': 'GitHub repository in owner/repo format (e.g. "facebook/r